In [ ]:
# Cell 1: Environment Setup
import os
import sys

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

# Set directories
if IS_KAGGLE:
    WORKING_DIR = '/kaggle/working'
    RESULTS_DIR = '/kaggle/working/results'
else:
    # Running locally
    WORKING_DIR = os.path.abspath('..')
    RESULTS_DIR = os.path.join(WORKING_DIR, 'results')

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Working directory: {WORKING_DIR}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
# Cell 2: Install Dependencies (Kaggle Only)
if IS_KAGGLE:
    # Install required dependencies
    print("📦 Installing dependencies...")
    !pip install -q transformers datasets plotly kaleido psutil scipy optuna mlflow
    print("✅ All dependencies installed!")
else:
    print("⚠️  Not on Kaggle - assuming dependencies are already installed")

In [ ]:
# Cell 3: GPU & Environment Check
import torch

print("=" * 60)
print("GPU Configuration")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    GPU_FLAG = "--kaggle-t4"
else:
    print("⚠️ No GPU available - training will be slower")
    GPU_FLAG = ""

print(f"\nPyTorch: {torch.__version__}")
print(f"Python: {sys.version}")

In [ ]:
# Cell 4: Configuration Summary

print("="*60)
print("🚀 PRODUCTION CONFIGURATION")
print("="*60)
print("Seeds: 10 seeds (42,123,456,789,1011,1213,1415,1617,1819,2021)")
print("Quick mode: False (full benchmark)")
print("Resume: True (safe to interrupt)")
print("Hyperparameter tuning: Enabled")
print("Experiments: ALL")
print("Profiling: Enabled")
print("Kaggle T4 optimizations: Enabled")
print("="*60)

In [ ]:
%%bash
# Cell 5: Run Complete Benchmark Suite
#
# This executes the full production benchmark with:
# ✅ 10 seeds for high statistical power
# ✅ Resume logic (safe to interrupt)
# ✅ Cross-experiment aggregation
# ✅ Statistical analysis (t-tests, effect sizes, power analysis)
# ✅ Hyperparameter tuning with Optuna
# ✅ Interactive visualizations
# ✅ Publication-ready reports
#

python /kaggle/input/gdsearch-repository/run_all_kaggle.py \
    --experiments all \
    --seeds 42,123,456,789,1011,1213,1415,1617,1819,2021 \
    --results-dir /kaggle/working/results \
    --kaggle-t4 \
    --profile \
    --resume

In [ ]:
# Cell 7: Display Results Summary
import pandas as pd
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

# List all result files
csv_files = list(results_path.rglob("*.csv"))
print(f"\n📁 Found {len(csv_files)} result files:\n")
for f in sorted(csv_files)[:20]:  # Show first 20
    print(f"  {f.relative_to(results_path)}")

if len(csv_files) > 20:
    print(f"  ... and {len(csv_files) - 20} more")

# Show cross-experiment aggregation if exists
agg_file = results_path / "analysis" / "cross_experiment_aggregation.csv"
if agg_file.exists():
    print("\n📊 Cross-Experiment Aggregation:")
    agg_df = pd.read_csv(agg_file)
    display(agg_df)

# Show optimizer rankings if exists
rank_file = results_path / "analysis" / "optimizer_rankings.csv"
if rank_file.exists():
    print("\n🏆 Optimizer Rankings:")
    rank_df = pd.read_csv(rank_file)
    display(rank_df)

In [ ]:
# Cell 8: Show Experiment-Specific Results

experiments_dir = results_path / "experiments"

if experiments_dir.exists():
    for exp_dir in sorted(experiments_dir.iterdir()):
        if exp_dir.is_dir():
            csv_files = list(exp_dir.glob("*.csv"))
            if csv_files:
                print(f"\n{'='*60}")
                print(f"📁 {exp_dir.name.upper()} RESULTS")
                print(f"{'='*60}")
                
                # Try to load and display main results
                for csv_file in sorted(csv_files)[:3]:  # Show first 3
                    try:
                        df = pd.read_csv(csv_file)
                        print(f"\n📄 {csv_file.name}")
                        print(f"   Shape: {df.shape}")
                        if 'optimizer' in df.columns or 'Optimizer' in df.columns:
                            opt_col = 'optimizer' if 'optimizer' in df.columns else 'Optimizer'
                            print(f"   Optimizers: {df[opt_col].unique().tolist()}")
                        display(df.head())
                    except Exception as e:
                        print(f"   Error reading: {e}")

In [ ]:
# Cell 9: Visualizations
import matplotlib.pyplot as plt

viz_dir = results_path / "visualizations"

if viz_dir.exists():
    # List available visualizations
    html_files = list(viz_dir.rglob("*.html"))
    png_files = list(viz_dir.rglob("*.png"))
    
    print(f"\n📈 Found {len(html_files)} interactive HTML plots")
    print(f"📊 Found {len(png_files)} static PNG plots")
    
    # Display some PNG plots
    for png_file in sorted(png_files)[:6]:  # Show first 6
        try:
            img = plt.imread(png_file)
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(png_file.stem)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not display {png_file.name}: {e}")
else:
    print("No visualizations directory found")

In [ ]:
# Cell 10: Statistical Analysis Summary

analysis_dir = results_path / "analysis"

if analysis_dir.exists():
    print("=" * 60)
    print("STATISTICAL ANALYSIS")
    print("=" * 60)
    
    # Show cross-experiment statistics
    stats_file = analysis_dir / "cross_experiment_statistics.csv"
    if stats_file.exists():
        print("\n🔬 Cross-Experiment Statistical Comparisons:")
        stats_df = pd.read_csv(stats_file)
        display(stats_df)
    
    # Show basic statistics
    basic_stats = analysis_dir / "00_basic_statistics.csv"
    if basic_stats.exists():
        print("\n📊 Basic Statistics:")
        basic_df = pd.read_csv(basic_stats)
        display(basic_df.head(20))
else:
    print("No analysis directory found")

In [ ]:
# Cell 11: Archive Results for Download
import shutil

if IS_KAGGLE:
    archive_name = f"gdsearch_results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
    archive_path = f"/kaggle/working/{archive_name}"
    
    print(f"Creating archive: {archive_name}.zip")
    shutil.make_archive(archive_path, 'zip', RESULTS_DIR)
    
    print(f"\n✅ Results archived to: {archive_path}.zip")
    print("   Download from Kaggle Output tab")
else:
    print("Not on Kaggle - results saved to:", RESULTS_DIR)

In [ ]:
# Cell 12: Quick Access Guide

print("""
=================================================================
📖 RESULTS QUICK ACCESS GUIDE
=================================================================

📁 Results Directory Structure:
   results/
   ├── experiments/           # Individual experiment data
   │   ├── mnist/            # MNIST benchmark results
   │   ├── cifar10/          # CIFAR-10 results
   │   ├── nlp/              # NLP (IMDB) results
   │   ├── resnet/           # ResNet18 results
   │   ├── highdim/          # High-dimensional functions
   │   ├── 2d/               # 2D test functions
   │   ├── robustness/       # Robustness analysis
   │   ├── sam/              # SAM sensitivity analysis
   │   └── ablation/         # Ablation studies
   │
   ├── analysis/              # Statistical analyses
   │   ├── cross_experiment_aggregation.csv    # Combined results
   │   ├── optimizer_rankings.csv              # Overall rankings
   │   ├── cross_experiment_statistics.csv     # Statistical tests
   │   ├── 00_basic_statistics.csv             # Basic stats
   │   ├── 01_convergence_rates.csv            # Convergence analysis
   │   └── 02_statistical_comparison.csv       # Pairwise comparisons
   │
   ├── visualizations/        # Plots and visualizations
   │   ├── interactive/       # Interactive HTML plots
   │   └── static/            # Static PNG/PDF plots
   │
   └── reports/               # Summary reports
       ├── experiment_summary_report.md
       └── 00_EXPERIMENT_SUMMARY.md

=================================================================

📊 Key Result Files:
   1. Cross-experiment aggregation:
      → analysis/cross_experiment_aggregation.csv
      
   2. Optimizer rankings (sorted by performance):
      → analysis/optimizer_rankings.csv
      
   3. Statistical significance tests:
      → analysis/cross_experiment_statistics.csv
      
   4. Convergence analysis:
      → analysis/01_convergence_rates.csv

=================================================================

🔬 Statistical Analysis Features:
   ✅ Multi-seed experiments (10 seeds for high statistical power)
   ✅ Student's t-tests for significance
   ✅ Cohen's d effect sizes
   ✅ Statistical power analysis
   ✅ Multiple comparison corrections:
      - Holm-Bonferroni (FWER control)
      - Benjamini-Hochberg (FDR control)

=================================================================

🔄 Resume Logic:
   • Uses --resume flag to skip completed experiments
   • Checks for existing CSV files before running
   • Safe to interrupt and restart at any time
   • Progress is saved incrementally

=================================================================

📈 Visualizations Available:
   • Training/test loss curves
   • Accuracy progression plots
   • Loss landscape 3D surfaces
   • Statistical comparison heatmaps
   • Convergence rate comparisons
   • Interactive parameter sensitivity plots

=================================================================

💾 Download Results:
   Kaggle: Results archived to .zip in /kaggle/working/
           Download from Output tab after completion
   
   Local: Results saved to: {RESULTS_DIR}

=================================================================

📖 For detailed documentation, see:
   • README.md in results directory
   • reports/experiment_summary_report.md
   • Individual experiment folders for per-run details

=================================================================
""".format(RESULTS_DIR=RESULTS_DIR))